# Installation

In [2]:
# https://docs.unsloth.ai/get-started/installing-+-updating/google-colab
# import os
# if "COLAB_" in "".join(os.environ.keys()):
#     # Do this only in Colab notebooks! Otherwise use pip install unsloth
#     !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#     !pip install --nodeps xformers "trl<0.9.0" peft accelerate bitsandbytes

### Check if CUDA is available

In [1]:
import torch

if torch.cuda.is_available():
  print("CUDA is available")
  print(torch.cuda.get_device_name(0))
else:
  print("CUDA is not available")


CUDA is available
Tesla T4


### Load the model for inference

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3",
    # model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    # model_name = "unsloth/phi-3.5-mini-instruct",
    # model_name = "unsloth/tinyllama-chat",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [4]:
FastLanguageModel.for_inference(model);

# Try it with a single example

In [9]:
system = '''
Re-phrase the given text in whole sentences as instructions to an undergraduate student. Don't use enumerations
or numbered paragraphs. Write continuous text. Pay attention to not forget any information.
'''

In [6]:
input = """
// The following sentence describes the configuration of a timelapse image acquisition workflow, which 
// is repeated at an interval of 10 minute(s) for a duration of 4 hour(s).
// Acquisition starts at a specific time, at 10:00.
// All the positions defined above are imaged.
// All the channels defined above are imaged.
// The plane distance, i.e. the z-step is 30.1 microns,
// the objective lens used is the 20x lens,
// in combination with the 0.5x magnification changer.
// Camera binning is set to 1 x 1 pixels (no binning).
At 10:00, acquire...
  every 10 minute(s) for 4 hour(s)
  all positions
  all channels
  with a plane distance of 30.1 microns
  using the 20x lens with the 0.5x magnification changer and a binning of 1 x 1.
"""

In [7]:
chat = [
    {"role": "system", "content": system},
    {"role": "user", "content": input}]

sample = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt = True)
# sample = prompt.format(system, input, "")
print(sample)



<s>[INST] 
Re-phrase the given text in whole sentences as instructions to an undergraduate student. Don't use enumerations
or numbered paragraphs. Write continuous text. Pay attention to not forget any information.



// The following sentence describes the configuration of a timelapse image acquisition workflow, which 
// is repeated at an interval of 10 minute(s) for a duration of 4 hour(s).
// Acquisition starts at a specific time, at 10:00.
// All the positions defined above are imaged.
// All the channels defined above are imaged.
// The plane distance, i.e. the z-step is 30.1 microns,
// the objective lens used is the 20x lens,
// in combination with the 0.5x magnification changer.
// Camera binning is set to 1 x 1 pixels (no binning).
At 10:00, acquire...
  every 10 minute(s) for 4 hour(s)
  all positions
  all channels
  with a plane distance of 30.1 microns
  using the 20x lens with the 0.5x magnification changer and a binning of 1 x 1.
[/INST]


In [9]:
%%time

# Use case             top_p       top_k     Description
# -----------------------------------------------------------------------------------
# Creative writing     0.92–0.97   50–100    Allows diverse, expressive outputs
# Conversational/chat  0.85–0.95   50–80     Keeps responses interesting but on-topic
# Safer, focused gen   0.8         20–50     Reduces drift or hallucination
# Highly controlled    0.6–0.8     10–20     Close to deterministic, very focused

# temperature: Lower  (e.g.       0.5) = safer, more conservative
#              Higher (e.g. 1.0 - 1.3) = more creative, more chaotic

nSamples = 1
inputs = tokenizer(sample, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, top_p=0.97, top_k=100, temperature=1.2, do_sample=True, num_return_sequences=nSamples)
# outputs = model.generate(**inputs, max_new_tokens=4096, use_cache=True, top_p=0.9, top_k=80, temperature=0.5, do_sample=True, num_return_sequences=nSamples)

for o in range(nSamples):
    full_output = tokenizer.decode(outputs[o], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    generated_text = full_output[len(prompt_text):].strip()
    print(generated_text)

To the undergraduate student,

In the timelapse image acquisition workflow, begin the process at 10:00. The acquisition should be repeated every 10 minutes for a duration of 4 hours. This means you should image all the defined positions and channels throughout this timeframe.

For the precise placement, or plane distance, use a z-step of 30.1 microns. During this process, employ the 20x lens, complemented with the 0.5x magnification changer. The camera binning should be set to 1 x 1 pixels, meaning no binning should be applied. Make sure to image all the channels and positions as defined.

Keep these instructions in mind when initiating the timelapse image acquisition process at the specified time.
CPU times: user 8.69 s, sys: 90.3 ms, total: 8.78 s
Wall time: 8.77 s


# Apply it to an entire dataset

### Load the dataset

In [5]:
from datasets import load_dataset

# For standard JSON formcat
dataset = load_dataset("json", data_files="autogenerated-sentences-with-context.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
print(dataset['train'][0]['context'])

// The following sentence describes the configuration of a channel, i.e. the settings needed 
// for exciting a specific fluorophore or for illuminating a brightfield image.
// The name of the channel is 'g0TyXG', the wavelength of the 
// light source (led or laser) is 567nm (nanometers), the intensity (power) of the 
// light source is 50%. The camera exposure time (illumination time) is 7232ms (milliseconds).
Define channel 'g0TyXG':
  excite with 50% at 567nm
  use an exposure time of 7232ms.




In [7]:
print(dataset['train'][0]['sentence'])

// The following sentence describes the configuration of a region, i.e. a cuboid within the sample 
// that is going to be imaged. 
// The name of the region is 'Q1G', its extent (i.e. dimensions) is given by its width 
// (30.837 micrometer), its height (226.531 micrometer) and its depth (460.6 micrometer).
// The center of the region is given by the three-dimensional stage position, which in this case is at 
// x = 190.491 micrometer, y = 599.181 micrometer and z = 503.437 micrometer.
Define a position 'Q1G':
  30.837 x 226.531 x 460.6 microns
  centered at (190.491, 599.181, 503.437) microns.


In [10]:
import re
import time
import os

if not os.path.exists('sentence-samples'):
    os.makedirs('sentence-samples')

# for i in range(1):
for i in range(0, len(dataset['train'])):
    l = dataset['train'][i]
    sentence = l['sentence']
    context = l['context']

    chat = [
        {"role": "system", "content": system},
        {"role": "user", "content": sentence},
    ]
    
    start = time.time()
    
    sample = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt = True)
    inputs = tokenizer(sample, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True, top_p=0.97, top_k=100, temperature=1, do_sample=True, num_return_sequences=1)

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    response = full_output[len(prompt_text):].strip()

    duration = time.time() - start
    print(f'Iteration {i}: {duration:.2f}')

    fi = open(f'sentence-samples/{i:04}-original.txt', 'w')
    fi.write(sentence)
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'w')
    fi.write(response)
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'w')
    fi.write(context)
    fi.close()



Iteration 0: 10.27
Iteration 1: 5.21
Iteration 2: 8.15
Iteration 3: 5.28
Iteration 4: 6.69
Iteration 5: 2.83
Iteration 6: 2.98
Iteration 7: 3.73
Iteration 8: 2.99
Iteration 9: 3.84
Iteration 10: 8.18
Iteration 11: 10.32
Iteration 12: 7.90
Iteration 13: 6.84
Iteration 14: 8.49
Iteration 15: 8.06
Iteration 16: 6.12
Iteration 17: 6.07
Iteration 18: 6.03
Iteration 19: 6.22
Iteration 20: 3.62
Iteration 21: 3.21
Iteration 22: 4.97
Iteration 23: 3.63
Iteration 24: 3.94
Iteration 25: 7.34
Iteration 26: 5.13
Iteration 27: 6.56
Iteration 28: 7.60
Iteration 29: 6.71
Iteration 30: 5.76
Iteration 31: 5.50
Iteration 32: 7.94
Iteration 33: 5.50
Iteration 34: 7.08
Iteration 35: 7.28
Iteration 36: 7.64
Iteration 37: 12.65
Iteration 38: 7.11
Iteration 39: 10.07
Iteration 40: 9.22
Iteration 41: 7.45
Iteration 42: 7.72
Iteration 43: 5.81
Iteration 44: 10.56
Iteration 45: 3.27
Iteration 46: 6.06
Iteration 47: 5.90
Iteration 48: 5.49
Iteration 49: 6.14
Iteration 50: 6.76
Iteration 51: 11.98
Iteration 52: 6.

## Split the samples in training and test data

In [11]:
import os

N = len(dataset['train'])
nTrain = int(0.8 * N)
nTest = N - nTrain

print("Split into", nTrain, "training samples and", nTest, "test samples") 

Split into 1600 training samples and 400 test samples


### Create the train dataset

In [12]:
import re

f = open("dataset-for-finetuning-sentences-train.json", "w")
f.write("")

for i in range(0, nTrain):
    fi = open(f'sentence-samples/{i:04}-original.txt', 'r')
    original = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'r')
    rephrased = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'r')
    context = fi.read()
    fi.close()

    original  = re.sub(r'[\r\n]', '\\\\n', original)
    context   = re.sub(r'[\r\n]', '\\\\n', context)
    rephrased = re.sub(r'[\r\n]', '\\\\n', rephrased)
    rephrased = rephrased.replace('"', "'")
    f.write('{"sentence": "' + original + '", "rephrased": "' + rephrased + '", "context": "' + context + '"}\n')
    f.flush()

f.close()

### Create the test dataset

In [13]:
f = open("dataset-for-finetuning-sentences-test.json", "w")
f.write("")

for i in range(nTrain, N):
    fi = open(f'sentence-samples/{i:04}-original.txt', 'r')
    original = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-rephrased.txt', 'r')
    rephrased = fi.read()
    fi.close()

    fi = open(f'sentence-samples/{i:04}-context.txt', 'r')
    context = fi.read()
    fi.close()

    original  = re.sub(r'[\r\n]', '\\\\n', original)
    context   = re.sub(r'[\r\n]', '\\\\n', context)
    rephrased = re.sub(r'[\r\n]', '\\\\n', rephrased)
    rephrased = rephrased.replace('"', "'")
    f.write('{"sentence": "' + original + '", "rephrased": "' + rephrased + '", "context": "' + context + '"}\n')
    f.flush()

f.close()